# 01 — Data Exploration

This notebook explores the audience targeting data across 6 platforms:
- **IAB categories**: Yahoo DSP, The Trade Desk (TTD), DV360
- **Social platforms**: TikTok, Snapchat, Meta
- **Meta & Yahoo JSONs**: Rich audience data with sizes and hierarchies
- **TTD Apps**: Top 1,000 apps ranked by category

Goal: Understand the naming variations, hierarchy structures, and the challenge of cross-platform matching.

In [ ]:
import sys
sys.path.insert(0, '..')

from collections import Counter, defaultdict
import pandas as pd
from data_loader import load_all, load_iab_csv, load_social_csv, load_meta_json, load_yahoo_json, load_ttd_apps, print_summary

In [ ]:
# Load all segments
segments = load_all()
print_summary(segments)

## Cross-Platform Naming Variations

The core challenge: the same audience concept has different names across platforms.

In [ ]:
# Show naming variations for key verticals
by_platform = defaultdict(list)
for s in segments:
    by_platform[s.platform].append(s)

keywords = ['auto', 'vehicle', 'car', 'pet', 'animal', 'fitness', 'sport']

for kw in keywords:
    print(f"\n{'='*60}")
    print(f"Keyword: '{kw}'")
    print(f"{'='*60}")
    for platform in sorted(by_platform):
        matches = [s for s in by_platform[platform] if kw in s.name.lower()][:5]
        if matches:
            print(f"\n  {platform}:")
            for m in matches:
                print(f"    - {m.name} ({' > '.join(m.hierarchy[:2])})")

In [ ]:
# Hierarchy depth distribution by platform
depth_data = []
for s in segments:
    depth_data.append({'platform': s.platform, 'depth': len(s.hierarchy), 'name': s.name})

df = pd.DataFrame(depth_data)
print("Hierarchy depth by platform:")
print(df.groupby('platform')['depth'].describe().round(1))

In [ ]:
# Segment type distribution
type_data = [{'platform': s.platform, 'type': s.segment_type} for s in segments]
df_types = pd.DataFrame(type_data)
print("\nSegment types by platform:")
for platform in sorted(df_types['platform'].unique()):
    types = df_types[df_types['platform'] == platform]['type'].value_counts()
    print(f"\n  {platform}:")
    for t, c in types.items():
        print(f"    {t}: {c}")

In [ ]:
# Audience size distribution (where available)
with_size = [s for s in segments if s.audience_size is not None]
print(f"Segments with audience size: {len(with_size)} / {len(segments)}")

size_df = pd.DataFrame([{
    'platform': s.platform,
    'name': s.name,
    'audience_size': s.audience_size,
    'size_millions': s.audience_size / 1_000_000
} for s in with_size])

print("\nAudience size stats by platform (millions):")
print(size_df.groupby('platform')['size_millions'].describe().round(1))